In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False
print(f"In colab: {IN_COLAB}")

In colab: True


In [2]:
if IN_COLAB:
  from pathlib import Path
  from google.colab import drive

  ROOT_PATH = Path('/content/drive/MyDrive/')
  OUTPUT_ROOT_PATH = ROOT_PATH / f"Colab_Data/gsm-symbolic/outputs"

  drive.mount(str(ROOT_PATH.parent))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
if IN_COLAB:
  import os

  DRIVE_PROJECT_PATH = ROOT_PATH / "Colab_Projects"
  REPO_FOLDER_NAME = 'gsm-symbolic-benchmarking'
  FULL_REPO_PATH = DRIVE_PROJECT_PATH / REPO_FOLDER_NAME

  import os
  os.chdir(FULL_REPO_PATH)
  !git pull origin main

  try:
    import gsm_benchmarker
  except ImportError:
    !pip install transformers datasets accelerate bitsandbytes numpy pandas tqdm psutil torch cloud-tpu-client  # other stuff probably missing too
    !pip install -e .
    os.kill(os.getpid(), 9)  # need to restart the session for import to be possible

  os.chdir("..")

From https://github.com/the-mysh/gsm-symbolic-benchmarking
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
import numpy as np
import logging
from huggingface_hub import login, whoami
import os

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.benchmark import BenchmarkRunner
from gsm_benchmarker.hf_model_evaluator import HuggingFaceModelEvaluator
from gsm_benchmarker.utils.logging_setup import install_colored_logger

if IN_COLAB:
  from google.colab import userdata
  api_token = userdata.get('HF_TOKEN')
else:
  api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']

login(api_token)

whoami()['name']


'the-mysh'

In [5]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py'):
    logging.getLogger(log_name).setLevel(logging.INFO)

logger = logging.getLogger(__name__)
install_colored_logger(level=logging.DEBUG)

In [6]:
open_models = [
    'google/gemma-2-9b-it',
    'google/gemma-2-2b-it',
    'meta-llama/Meta-Llama-3-8B-Instruct',
    'microsoft/Phi-3.5-mini-instruct',
    'microsoft/Phi-3-medium-128k-instruct',
    'microsoft/Phi-3-small-128k-instruct',
    'microsoft/phi-2'
    'mistralai/Mathstral-7B-v0.1',
    'mistralai/Mistral-7B-Instruct-v0.1',
]

In [7]:
br = BenchmarkRunner(
    models=open_models,
    dset_variants=[GSMSymbolicDataset.Variant.main], #, GSMSymbolicDataset.Variant.p1, GSMSymbolicDataset.Variant.p2],
    storage_path=OUTPUT_ROOT_PATH,
    config=BenchmarkConfig(trust_remote_code=True)
)

In [ ]:
br.run(n_sets=3, n_per_set=4)

gsm_benchmarker.benchmark [INFO] 2025-10-17 10:30:51: ========== Evaluating model 1/8: google/gemma-2-9b-it ==========
gsm_benchmarker.hf_model_wrapper [INFO] 2025-10-17 10:30:51: Setting up model google/gemma-2-9b-it
gsm_benchmarker.hf_model_wrapper [DEBUG] 2025-10-17 10:30:51: Loading tokeniser


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

In [ ]:
br.summarise_results()